In [1]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
from sklearn.pipeline import Pipeline
import optuna
import joblib 

In [2]:
df = pd.read_csv("used_cars_in_egypt.csv")
df.head()

,Brand,Model,Color,Year,Fuel,Kilometers,Trasmission,Price,Gov
0,BMW,525,Black,2001.0,gas,"255,000 Km",automatic,"380,000 EGP",Heliopolis
1,Opel,Astra,Bronze,2015.0,gas,"130,000 Km",automatic,"480,000 EGP",Cairo
2,Chevrolet,Optra,Dark blue,2022.0,gas,"50,000 Km",automatic,"460,000 EGP",Cairo
3,Mitsubishi,Lancer Puma,Black,2009.0,gas,"165,000 Km",automatic,"295,000 EGP",Alexandria
4,Mercedes,E 320,Cyan,2004.0,gas,"300,000 Km",automatic,"580,000 EGP",Nasr city


# Dataset Column Description – Used Cars in Egypt

## Column Descriptions

| Column | Description |
|---|---|
| Brand | Car manufacturer (e.g. Toyota, Hyundai, Nissan, BMW, Mercedes) — 94 brands. |
| Model | Car model within the brand (e.g. Corolla, Elantra, Sunny) — 828 models. |
| Color | Exterior color of the car (e.g. Black, White, Silver, Gray). |
| Year | Manufacturing year of the car (1951 – 2023). |
| Fuel | Fuel type: gas (petrol), diesel, natural gas, Electric, or Hybrid. |
| Kilometers | Distance the car has been driven (stored as text, e.g. "130,000 Km"). |
| Trasmission | Transmission type (automatic / manual / CVT). *(column name is misspelled in the source)* |
| Price | Asking price of the car in Egyptian Pounds (stored as text, e.g. "480,000 EGP"). **Target variable.** |
| Gov | Location of the listing — governorate, city or district (e.g. Cairo, Alexandria, Nasr city). |


# Data Review 

In [3]:
df.columns=df.columns.str.replace("Trasmission","Transmission")

In [4]:
df.columns=df.columns.str.strip().str.lower().str.replace(" ","_")

In [5]:
df.head()

,brand,model,color,year,fuel,kilometers,transmission,price,gov
0,BMW,525,Black,2001.0,gas,"255,000 Km",automatic,"380,000 EGP",Heliopolis
1,Opel,Astra,Bronze,2015.0,gas,"130,000 Km",automatic,"480,000 EGP",Cairo
2,Chevrolet,Optra,Dark blue,2022.0,gas,"50,000 Km",automatic,"460,000 EGP",Cairo
3,Mitsubishi,Lancer Puma,Black,2009.0,gas,"165,000 Km",automatic,"295,000 EGP",Alexandria
4,Mercedes,E 320,Cyan,2004.0,gas,"300,000 Km",automatic,"580,000 EGP",Nasr city


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17159 entries, 0 to 17158
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         17159 non-null  str    
 1   model         17159 non-null  str    
 2   color         17159 non-null  str    
 3   year          17157 non-null  float64
 4   fuel          17159 non-null  str    
 5   kilometers    17159 non-null  str    
 6   transmission  17159 non-null  str    
 7   price         17159 non-null  str    
 8   gov           17159 non-null  str    
dtypes: float64(1), str(8)
memory usage: 2.1 MB


In [7]:
df.columns.to_list()

['brand',
 'model',
 'color',
 'year',
 'fuel',
 'kilometers',
 'transmission',
 'price',
 'gov']

In [8]:
df.isna().sum().sort_values(ascending=False)

year            2
brand           0
model           0
color           0
fuel            0
kilometers      0
transmission    0
price           0
gov             0
dtype: int64

# Missing Values

In [9]:
df=df.dropna(subset=["year"])

In [10]:
df.isna().sum().sort_values(ascending=False)

brand           0
model           0
color           0
year            0
fuel            0
kilometers      0
transmission    0
price           0
gov             0
dtype: int64

# Remove Duplicates

In [11]:
df.duplicated().sum()

np.int64(272)

In [12]:
df=df.drop_duplicates()

In [13]:
df.duplicated().sum()

np.int64(0)

# Data Cleaning

## Fix Data Types

In [14]:
num_edit=['price','kilometers']
for col in num_edit:
     print(col)
     print(f"  nunique: {df[col].nunique()}")
     print(f"  unique:  {df[col].unique().tolist()}")
     print("-" * 50)
for col in num_edit:
    df[col] = df[col].str.replace(",", "").str.extract(r"(\d+)")[0].astype(float)

price
  nunique: 716
  unique:  ['380,000 EGP', '480,000 EGP', '460,000 EGP', '295,000 EGP', '580,000 EGP', '290,000 EGP', '470,000 EGP', '630,000 EGP', '399,000 EGP', '550,000 EGP', '60,000 EGP', '45,000 EGP', '855,000 EGP', '900,000 EGP', '450,000 EGP', '75,000 EGP', '490,000 EGP', '40,000 EGP', '370,000 EGP', '136,000 EGP', '330,000 EGP', '130,000 EGP', '105,000 EGP', '670,000 EGP', '125,000 EGP', '138,000 EGP', '620,000 EGP', '395,000 EGP', '1,750,000 EGP', '310,000 EGP', '540,000 EGP', '700,000 EGP', '18,000 EGP', '220,000 EGP', '335,000 EGP', '420,000 EGP', '360,000 EGP', '625,000 EGP', '160,000 EGP', '650,000 EGP', '920,000 EGP', '525,000 EGP', '800,000 EGP', '50,000 EGP', '475,000 EGP', '680,000 EGP', '509,000 EGP', '510,000 EGP', '355,000 EGP', '65,000 EGP', '0 EGP', '1,100,000 EGP', '190,000 EGP', '375,000 EGP', '85,000 EGP', '175,000 EGP', '198,000 EGP', '260,000 EGP', '300,000 EGP', '170,000 EGP', '280,000 EGP', '140,000 EGP', '53,000 EGP', '70,000 EGP', '400,000 EGP', '120

In [15]:
for col in num_edit:
    print(col)
    print(f"  nunique: {df[col].nunique()}")
    print(f"  unique:  {df[col].unique().tolist()}")
    print("-" * 50)

price
  nunique: 716
  unique:  [380000.0, 480000.0, 460000.0, 295000.0, 580000.0, 290000.0, 470000.0, 630000.0, 399000.0, 550000.0, 60000.0, 45000.0, 855000.0, 900000.0, 450000.0, 75000.0, 490000.0, 40000.0, 370000.0, 136000.0, 330000.0, 130000.0, 105000.0, 670000.0, 125000.0, 138000.0, 620000.0, 395000.0, 1750000.0, 310000.0, 540000.0, 700000.0, 18000.0, 220000.0, 335000.0, 420000.0, 360000.0, 625000.0, 160000.0, 650000.0, 920000.0, 525000.0, 800000.0, 50000.0, 475000.0, 680000.0, 509000.0, 510000.0, 355000.0, 65000.0, 0.0, 1100000.0, 190000.0, 375000.0, 85000.0, 175000.0, 198000.0, 260000.0, 300000.0, 170000.0, 280000.0, 140000.0, 53000.0, 70000.0, 400000.0, 120000.0, 350000.0, 1150000.0, 26000.0, 250000.0, 780000.0, 1400000.0, 385000.0, 415000.0, 500000.0, 110000.0, 770000.0, 320000.0, 2150000.0, 1580000.0, 405000.0, 115000.0, 235000.0, 135000.0, 95000.0, 760000.0, 90000.0, 410000.0, 155000.0, 455000.0, 1775000.0, 465000.0, 875000.0, 230000.0, 980000.0, 570000.0, 345000.0, 1500000.

In [16]:
df["year"] = df["year"].astype(int)

In [17]:
df.info()

<class 'pandas.DataFrame'>
Index: 16885 entries, 0 to 17149
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         16885 non-null  str    
 1   model         16885 non-null  str    
 2   color         16885 non-null  str    
 3   year          16885 non-null  int64  
 4   fuel          16885 non-null  str    
 5   kilometers    16885 non-null  float64
 6   transmission  16885 non-null  str    
 7   price         16885 non-null  float64
 8   gov           16885 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 1.9 MB


## Clean Text Columns

In [18]:
for col in df.select_dtypes(exclude="number").columns:
    print(col)
    print(f"   nunique: {df[col].nunique()}")
    print(f"   unique:  {df[col].unique().tolist()}")
    print("-" * 50)

brand
   nunique: 94
   unique:  ['BMW', 'Opel', 'Chevrolet', 'Mitsubishi', 'Mercedes', 'Honda', 'MG', 'Nissan', 'Mazda', 'Skoda', 'Chery', 'Hyundai', 'Lada', 'Kia', 'Daewoo', 'Fiat', 'Daihatsu', 'Seat', 'Speranza', 'Volkswagen', 'Jac', 'Brilliance', 'Suzuki', 'Ssang Yong', 'Toyota', 'Renault', 'Byd', 'Jetour', 'Peugeot', 'Jeep', 'Citroën', 'Alfa Romeo', 'Dodge', 'DFSK', 'Geely', 'Proton', 'Mahindra', 'Isuzu', 'Ford', 'Canghe || Changhe', 'Faw', 'Haval', 'Mini', 'Foton', 'Senova', 'Zotye', 'Karry', 'Volvo', 'Changan', 'Land Rover', 'Gmc', 'Subaru', 'Saipa', 'Baic', 'Infiniti', 'Great Wall', 'Kenbo', 'Dongfeng', 'Ds', 'Jaguar', 'Hawtai', 'Audi', 'Hafei', 'Chrysler', 'Datsun', 'Lincoln', 'Emgrand', 'Victory', 'Soueast', 'Lifan', 'Lancia', 'Chana', 'Bestune', 'Cupra', 'Haima', 'Tesla', 'Porsche', 'Landwind', 'JMC', 'Kaiyi', 'Cadillac', 'Aston Martin', 'Smart', 'Maserati', 'Pontiac', 'Canghe  || Changhe', 'Exeed', 'Buick', 'Hummer', 'Bentley', 'Sokon', 'Lexus', 'Keyton', 'Perodua']
-------

In [19]:
df["brand"] = df["brand"].str.replace(r"\s+", " ", regex=True).str.strip()

In [20]:
df["model"] = df["model"].str.replace(r"\s+", " ", regex=True).str.strip()

In [21]:
df["model"] = df["brand"] + " " + df["model"]

In [22]:
df["gov"] = df["gov"].replace("6-Oct", "6 October")

## Price Outliers

In [23]:
df.describe()

,year,kilometers,price
count,16885.000000,1.688500e+04,1.688500e+04
mean,2011.270951,1.181859e+05,4.918885e+05
std,10.500571,9.135132e+04,7.710115e+06
min,1951.000000,0.000000e+00,0.000000e+00
25%,2007.000000,4.400000e+04,1.550000e+05
50%,2014.000000,1.080000e+05,3.120000e+05
75%,2019.000000,1.800000e+05,5.100000e+05
max,2023.000000,1.400000e+06,1.000000e+09


In [24]:
(df["price"] == 0).sum()

np.int64(36)

In [25]:
df = df[df["price"] != 0]

In [26]:
df[df["price"] <= 10_000]

,brand,model,color,year,fuel,kilometers,transmission,price,gov
278,Alfa Romeo,Alfa Romeo 156,Red,2000,gas,1.0,manual,1.0,6 October
1298,Peugeot,Peugeot 508,Dark blue,2013,gas,178000.0,automatic,1.0,Port Said
1674,Daewoo,Daewoo Lanos,Dark blue,2000,gas,0.0,automatic,1.0,Sohag
2827,Speranza,Speranza A516,White,2010,gas,330000.0,automatic,1.0,Imbaba
4018,Hyundai,Hyundai Elantra HD,Silver,2021,gas,40000.0,automatic,1.0,Cairo
5738,Speranza,Speranza A11,Red,2023,gas,50000.0,automatic,1.0,Giza
6819,Hyundai,Hyundai Tucson GDI,White,2017,gas,125000.0,automatic,700.0,Sohag
10036,Gmc,Gmc Yukon,Gold,2007,gas,165000.0,automatic,1.0,Alexandria
10179,Gmc,Gmc Yukon,Black,2017,gas,124000.0,automatic,1.0,Cairo
12170,Mercedes,Mercedes 280,Black,2005,gas,170000.0,automatic,1000.0,Alexandria


In [27]:
df = df[df["price"] >= 10_000]

In [28]:
df[df["price"] > 10_000_0000]

,brand,model,color,year,fuel,kilometers,transmission,price,gov
14522,Skoda,Skoda Octavia A4,Silver,2000,gas,250000.0,manual,1.000000e+09,Sheikh Zayed City


In [29]:
df = df[df["price"] < 100_000_000]

In [30]:
df[df["price"] >= df["price"].quantile(0.999)]

,brand,model,color,year,fuel,kilometers,transmission,price,gov
1830,Mercedes,Mercedes AMG GT,Gray,2021,gas,0.0,automatic,5900000.0,Heliopolis
2465,Mercedes,Mercedes 500,Black,2022,gas,1000.0,automatic,10000000.0,Tagamo3 - New Cairo
2588,Mercedes,Mercedes EQS,Black,2022,Electric,40.0,automatic,7500000.0,6 October
2828,Mercedes,Mercedes S 560,Silver,2018,gas,20000.0,automatic,6500000.0,Heliopolis
3407,Land Rover,Land Rover Range Rover Sport || Sport,Black,2022,gas,5000.0,automatic,7000000.0,Cairo
3458,Mercedes,Mercedes G Class,Gold,2023,gas,100.0,automatic,10000000.0,Tagamo3 - New Cairo
3459,BMW,BMW I7,Silver,2023,Electric,100.0,automatic,6000000.0,Tagamo3 - New Cairo
3469,Mercedes,Mercedes S 450,Black,2022,gas,900.0,automatic,6300000.0,Sheikh Zayed City
3583,BMW,BMW I8,Silver,2023,Electric,10.0,automatic,7900000.0,Tagamo3 - New Cairo
3596,BMW,BMW I8,White,2023,Electric,100.0,automatic,6300000.0,Tagamo3 - New Cairo


## Kilometers Outliers

In [31]:
(df["kilometers"] == 0).sum()

np.int64(1442)

In [32]:
df[df["kilometers"] > 1_000_000]

,brand,model,color,year,fuel,kilometers,transmission,price,gov
14738,Kia,Kia Cerato,Black,2014,gas,1400000.0,automatic,420000.0,Mohandessin


In [33]:
df=df[df["kilometers"] < 1_000_000]

In [34]:
df.describe(exclude="number")

,brand,model,color,fuel,transmission,gov
count,16830,16830,16830,16830,16830,16830
unique,93,868,24,5,3,102
top,Hyundai,Nissan Sunny,Silver,gas,automatic,Cairo
freq,1772,794,3293,16209,10815,2378


In [35]:
latest_year = df["year"].max()
age = latest_year - df["year"] + 1

km_per_year = df["kilometers"] / age
df = df[km_per_year <= 100_000]


In [36]:
df = df.reset_index(drop=True)

# Feature Engineering

In [37]:
df["model_freq"] = df["model"].map(df["model"].value_counts())

In [38]:
df["bm_first_year"] = df.groupby("model")["year"].transform("min")

In [39]:
df["years_since_first_seen"] = df["year"] - df["bm_first_year"]

In [40]:
df.to_csv("car_usage_clean.csv", index=False)

# EDA CAT

In [41]:
for col in df.select_dtypes(exclude="number").columns:
    stats = df.groupby(col)["price"].agg(median="median", count="count")
    stats = stats.sort_values("count", ascending=False).head(15).sort_values("median")
    fig = px.bar(stats, x=stats.index, y="median", text="count",
                 title=f"Median Price by {col}",
                 color_discrete_sequence=["#57B3E1"])
    fig.update_traces(texttemplate="n=%{text:,}", textposition="outside")
    fig.update_layout(yaxis_title="Median Price (EGP)", xaxis_title=col,
                      yaxis_tickformat="~s", title_x=0.5)
    fig.update_yaxes(range=[0, stats["median"].max() * 1.15])
    fig.show()

## EDA Summary for Categorical Features
- **brand:** The strongest pattern. Prices rise in clear steps by brand, from Fiat (median 50K EGP) to Mercedes (895K), almost 18x. Part of this is car age: the median Fiat in the data is from 1996, while the median Mercedes is from 2013.
- **transmission:** Automatic cars (430K) cost more than 3x as much as manual cars (130K). CVT looks the most expensive (555K), but it has only 6 cars, so this number is not reliable.
- **fuel:** Most cars run on gas (median price 320K). Electric and hybrid cars cost about 6 times more (around 2M), mainly because they are newer models. However, there are very few of them (47 and 18 cars), so this result is less reliable. Natural gas cars are the cheapest (170K).
- **gov:** Cars in wealthier areas are more expensive. Sheikh Zayed (550K) and New Cairo (502K) are about twice as expensive as Helwan and Shobra (242K).
- **model:** Ranges from the Fiat 128 (27.5K) to the Kia Sportage (905K). This mostly repeats the brand pattern at a finer level.
- **color:** Weak effect. Most common colors fall between 280K and 390K. Green (115K) is the cheapest, but this is because green cars are old (median year 1998), not because of the color itself.

# EDA for Num

In [42]:
for col in df.select_dtypes(include="number").columns.drop("price"):
    data = df
    if col == "kilometers":
        data = df[df["kilometers"] > 0]   # zero km = mostly missing values (old cheap cars), hide them in this plot only
    bins = pd.qcut(data[col], q=10, duplicates="drop")
    stats = data.groupby(bins, observed=True)["price"].agg(median="median", count="count")
    fmt = "{:,.0f}" if data[col].max() >= 10_000 else "{:.0f}"
    stats.index = [f"{fmt.format(max(iv.left, data[col].min()))} - {fmt.format(iv.right)}" for iv in stats.index]
    fig = px.bar(stats, x=stats.index, y="median", text="count",
                 title=f"Median Price by {col}",
                 color_discrete_sequence=["#57B3E1"])
    fig.update_traces(texttemplate="n=%{text:,}", textposition="outside")
    fig.update_layout(yaxis_title="Median Price (EGP)", xaxis_title=col,
                      yaxis_tickformat="~s", title_x=0.5)
    fig.update_yaxes(range=[0, stats["median"].max() * 1.15])
    fig.show()

## EDA Summary for Numerical Features
- **year:** The strongest pattern. Price rises steadily with every newer group of years, from 55K for cars made before 1997 to 625K for 2021-2023 cars, more than 11x. Newer cars are always more expensive.
- **kilometers:** Price drops as kilometers go up, from 650K for cars under 24,000 km to 180K for cars over 240,000 km. The drop is clear but smaller than the year effect, and part of it is age, because older cars usually have more kilometers. The 1,442 cars with 0 km are hidden in this plot, because 0 here mostly means the value was not entered.
- **model_freq:** No clear pattern on its own. Prices go up and down (225K to 410K) with no direction, because both popular and rare models can be cheap or expensive. It still improved the model in testing, because it helps the model when combined with the other features.
- **bm_first_year:** Models that first appear recently in the data are more expensive. Models that first appear after 2017 have a median of 650K, compared with about 150K-160K for models that first appear before 1997. This mostly reflects newer models replacing older ones.
- **years_since_first_seen:** Weak pattern. Price stays around 250K-320K for most values and rises to about 400K for cars more than 21 years after the model first appears. These are usually new versions of long-running models such as Mercedes and Toyota Corolla, so they are newer cars.

# Detect Outliers

In [43]:
from datasist.structdata import detect_outliers
for col in df.select_dtypes(include="number").columns.to_list():
  outliers=detect_outliers(data=df.dropna(subset=[col]),features=[col],n=0)
  print(f" {col}:")
  print(f"    Outliers Count: {len(outliers)}")
  print(f"    Outliers Percentage: {len(outliers) / len(df) * 100:.2f}%")
  print("-" * 50)

 year:
    Outliers Count: 878
    Outliers Percentage: 5.22%
--------------------------------------------------
 kilometers:
    Outliers Count: 151
    Outliers Percentage: 0.90%
--------------------------------------------------
 price:
    Outliers Count: 1069
    Outliers Percentage: 6.36%
--------------------------------------------------
 model_freq:
    Outliers Count: 1270
    Outliers Percentage: 7.55%
--------------------------------------------------
 bm_first_year:
    Outliers Count: 199
    Outliers Percentage: 1.18%
--------------------------------------------------
 years_since_first_seen:
    Outliers Count: 936
    Outliers Percentage: 5.57%
--------------------------------------------------


# Split Data

In [44]:
from sklearn.model_selection import train_test_split

x = df.drop(columns="price")
y = df["price"]

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

numeric = x.select_dtypes(include="number").columns.to_list()
te_cat = ["brand", "model", "gov"]
binary_cat = ["color"]
ohe_cat = ["fuel", "transmission"]




# Preprocessing

In [45]:
df.isna().sum()

brand                     0
model                     0
color                     0
year                      0
fuel                      0
kilometers                0
transmission              0
price                     0
gov                       0
model_freq                0
bm_first_year             0
years_since_first_seen    0
dtype: int64

# OHE Pipeline

In [46]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False,handle_unknown="ignore")
ohe_pipeline=Pipeline(steps=[("ohe",ohe)])
display(ohe_pipeline)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories

# Binary Pipeline

In [47]:
from category_encoders import BinaryEncoder
binary=BinaryEncoder()
binary_pipeline=Pipeline(steps=[("binary",binary)])
display(binary_pipeline)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('binary', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,verbose,0
,cols,None
,mapping,None
,drop_invariant,False
,return_df,True
,base,2
,handle_unknown,'value'


# Target Encoding Pipeline

In [48]:
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import KFold
te=TargetEncoder(target_type="continuous", cv=KFold(n_splits=5, shuffle=True, random_state=42))
te_pipeline=Pipeline(steps=[("te",te)])
display(te_pipeline)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('te', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"target_type target_type: {""auto"", ""continuous"", ""binary"", ""multiclass""}, default=""auto""Type of target.- `""auto""` : Type of target is inferred with :func:`~sklearn.utils.multiclass.type_of_target`.- `""continuous""` : Continuous target- `""binary""` : Binary target- `""multiclass""` : Multiclass target.. note:: The type of target inferred with `""auto""` may not be the desired target type used for modeling. For example, if the target consisted of integers between 0 and 100, then :func:`~sklearn.utils.multiclass.type_of_target` will infer the target as `""multiclass""`. In this case, setting `target_type=""continuous""` will specify the target as a regression problem. The `target_type_` attribute gives the target type used by the encoder... versionchanged:: 1.4 Added the option 'multiclass'.",'continuous'
,"cv cv: int, cross-validation generator or an iterable, default=5Determines the splitting strategy used in the internal :term:`cross fitting`during :meth:`fit_transform`. Splitters where each sample index doesn't appearin the validation fold exactly once, raise a `ValueError`.Possible inputs for cv are:- `None`, to use a 5-fold cross-validation chosen internally based on `target_type`,- integer, to specify the number of folds for the cross-validation chosen internally based on `target_type`,- :term:`CV splitter` that does not repeat samples across validation folds,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if `target_type` is `""continuous""`, :class:`KFold` isused, otherwise :class:`StratifiedKFold` is used.Refer :ref:`User Guide <cross_validation>` for more information oncross-validation strategies... versionchanged:: 1.9 Cross-validation generators and iterables can also be passed as `cv`.",KFold(n_split... shuffle=True)
,"categories categories: ""auto"" or list of shape (n_features,) of array-like, default=""auto""Categories (unique values) per feature:- `""auto""` : Determine categories automatically from the training data.- list : `categories[i]` holds the categories expected in the i-th column. The passed categories should not mix strings and numeric va

# Column Transformer

In [49]:
from sklearn.compose import ColumnTransformer
preprocessing=ColumnTransformer(transformers=[("ohe",ohe_pipeline,ohe_cat),
                                              ("binary",binary_pipeline,binary_cat),
                                              ("te",te_pipeline,te_cat)
                                              ],remainder="passthrough",verbose_feature_names_out=False)
display(preprocessing)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per t

# Models Comparison R² 

In [50]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, KFold

models=[("XGBRegressor",XGBRegressor(random_state=42)),
        ("LGBMRegressor",LGBMRegressor(random_state=42,verbose=-1)),
        ("CatBoostRegressor",CatBoostRegressor(random_state=42,verbose=0))]

cv=KFold(shuffle=True,n_splits=5,random_state=42)

for model in models:
    model_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",model[1])])

    result=cross_validate(model_pipeline,x_train,y_train,cv=cv,return_train_score=True,n_jobs=1,scoring="r2")

    print(model[0])
    print(f"train_score {result["train_score"].mean()*100 :.2f}")
    print(f"test_score {result["test_score"].mean()*100 :.2f}")
    print("-" * 50)

XGBRegressor
train_score 90.60
test_score 80.00
--------------------------------------------------
LGBMRegressor
train_score 89.80
test_score 79.46
--------------------------------------------------
CatBoostRegressor
train_score 91.33
test_score 80.59
--------------------------------------------------


# Models Comparison with Log Target R² 

In [51]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import TransformedTargetRegressor

models=[("XGBRegressor",XGBRegressor(random_state=42)),
        ("LGBMRegressor",LGBMRegressor(random_state=42,verbose=-1)),
        ("CatBoostRegressor",CatBoostRegressor(random_state=42,verbose=0))]

cv=KFold(n_splits=5,shuffle=True,random_state=42)

for model in models:
    model_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",model[1])])
    log_model=TransformedTargetRegressor(regressor=model_pipeline,func=np.log1p,inverse_func=np.expm1)

    result=cross_validate(log_model,x_train,y_train,cv=cv,return_train_score=True,n_jobs=1,
                          scoring="r2")

    print(model[0])
    print(f"train_score {result["train_score"].mean()*100 :.2f}")
    print(f"test_score {result["test_score"].mean()*100 :.2f}")
    print("-" * 50)

XGBRegressor
train_score 91.43
test_score 78.88
--------------------------------------------------
LGBMRegressor
train_score 89.63
test_score 80.61
--------------------------------------------------
CatBoostRegressor
train_score 91.03
test_score 81.04
--------------------------------------------------


# RandomizedSearch (R² EGP)

In [52]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

param_grid = {
    "regressor__model__depth": randint(4, 9),
    "regressor__model__learning_rate": loguniform(0.01, 0.2),
    "regressor__model__iterations": randint(500, 2001),
    "regressor__model__l2_leaf_reg": loguniform(1, 10),
    "regressor__model__random_strength": uniform(0, 10),
}

cat_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",CatBoostRegressor(random_state=42,verbose=0))])
log_model=TransformedTargetRegressor(regressor=cat_pipeline,func=np.log1p,inverse_func=np.expm1)

search=RandomizedSearchCV(log_model,param_grid,scoring="r2",n_iter=20,cv=cv,return_train_score=True,n_jobs=1,random_state=42)
search.fit(x_train,y_train)

print(f"Best_param: {search.best_params_}")
print(f"Best_score: {search.best_score_*100:.2f}")




Best_param: {'regressor__model__depth': 7, 'regressor__model__iterations': 776, 'regressor__model__l2_leaf_reg': np.float64(4.144589378013023), 'regressor__model__learning_rate': np.float64(0.06248547977119089), 'regressor__model__random_strength': np.float64(0.07066305219717406)}
Best_score: 81.13


# RandomizedSearch (R² log)

In [53]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import make_scorer, r2_score

def r2_log(y_true, y_pred):
    return r2_score(np.log1p(y_true), np.log1p(y_pred))

r2_log_scorer = make_scorer(r2_log)

param_grid = {
    "regressor__model__depth": randint(4, 9),
    "regressor__model__learning_rate": loguniform(0.01, 0.2),
    "regressor__model__iterations": randint(500, 2001),
    "regressor__model__l2_leaf_reg": loguniform(1, 10),
    "regressor__model__random_strength": uniform(0, 10),
}

cat_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",CatBoostRegressor(random_state=42,verbose=0))])
log_model=TransformedTargetRegressor(regressor=cat_pipeline,func=np.log1p,inverse_func=np.expm1)

search=RandomizedSearchCV(log_model,param_grid,scoring=r2_log_scorer,n_iter=20,cv=cv,return_train_score=True,n_jobs=1,random_state=42)
search.fit(x_train,y_train)

print(f"Best_param: {search.best_params_}")
print(f"Best_score: {search.best_score_*100:.2f}")

Best_param: {'regressor__model__depth': 7, 'regressor__model__iterations': 776, 'regressor__model__l2_leaf_reg': np.float64(4.144589378013023), 'regressor__model__learning_rate': np.float64(0.06248547977119089), 'regressor__model__random_strength': np.float64(0.07066305219717406)}
Best_score: 93.41


# Optuna (R² log)

In [54]:
import optuna
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

def objective(trial):
    depth = trial.suggest_int("depth", 4, 8)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
    iterations = trial.suggest_int("iterations", 500, 2000)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1, 10, log=True)
    random_strength = trial.suggest_float("random_strength", 0, 10)

    model = clone(log_model)
    model.set_params(regressor__model__depth=depth,
                     regressor__model__learning_rate=learning_rate,
                     regressor__model__iterations=iterations,
                     regressor__model__l2_leaf_reg=l2_leaf_reg,
                     regressor__model__random_strength=random_strength,
                     regressor__model__thread_count=8)

    score = cross_val_score(model, x_train, y_train, cv=cv, scoring=r2_log_scorer, n_jobs=1)
    return score.mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print(f"Best_param: {study.best_params}")
print(f"Best_score: {study.best_value*100:.2f}")

[I 2026-11-10 03:41:43,284] A new study created in memory with name: no-name-f5c1bef0-8f13-4851-b1ee-b7330d66e484
[I 2026-11-10 03:41:54,109] Trial 0 finished with value: 0.9327331428594313 and parameters: {'depth': 5, 'learning_rate': 0.17254716573280354, 'iterations': 1598, 'l2_leaf_reg': 3.968793330444372, 'random_strength': 1.5601864044243652}. Best is trial 0 with value: 0.9327331428594313.
[I 2026-11-10 03:42:04,458] Trial 1 finished with value: 0.9230711666764078 and parameters: {'depth': 4, 'learning_rate': 0.011900590783184251, 'iterations': 1800, 'l2_leaf_reg': 3.9913058785616786, 'random_strength': 7.080725777960454}. Best is trial 0 with value: 0.9327331428594313.
[I 2026-11-10 03:42:14,469] Trial 2 finished with value: 0.9312467046170905 and parameters: {'depth': 4, 'learning_rate': 0.18276027831785724, 'iterations': 1749, 'l2_leaf_reg': 1.6305687346221471, 'random_strength': 1.8182496720710062}. Best is trial 0 with value: 0.9327331428594313.
[I 2026-11-10 03:42:21,968] T

Best_param: {'depth': 7, 'learning_rate': 0.054875377919154984, 'iterations': 1571, 'l2_leaf_reg': 6.534791492572896, 'random_strength': 9.460659097449298}
Best_score: 93.45


# Best Model

In [55]:
best_cat = CatBoostRegressor(depth=7,
                             learning_rate=0.054875377919154984,
                             iterations=1571,
                             l2_leaf_reg=6.534791492572896,
                             random_strength=9.460659097449298,
                             verbose=0, random_state=42, thread_count=8)

cat_boost_pipeline = Pipeline(steps=[("preprocessing", preprocessing), ("model", best_cat)])
final_model = TransformedTargetRegressor(regressor=cat_boost_pipeline, func=np.log1p, inverse_func=np.expm1)
final_model.fit(x_train, y_train)
result = cross_validate(final_model, x_train, y_train, cv=cv, scoring=r2_log_scorer,
                        return_train_score=True, n_jobs=1)
print(f"Train_score: {result['train_score'].mean()*100:.2f}")
print(f"Test_score:  {result['test_score'].mean()*100:.2f}")

Train_score: 96.48
Test_score:  93.45


# Test Set Evaluation

In [56]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = final_model.predict(x_test)

ape = np.abs(y_test - y_pred) / y_test
print(f"R2 (log):  {r2_score(np.log1p(y_test), np.log1p(y_pred))*100:.2f}")
print(f"R2 (EGP):  {r2_score(y_test, y_pred)*100:.2f}")
print(f"MAE:       {mean_absolute_error(y_test, y_pred):,.0f} EGP")
print(f"MdAPE:     {np.median(ape)*100:.2f}%")

R2 (log):  94.58
R2 (EGP):  81.84
MAE:       69,425 EGP
MdAPE:     11.42%


# Feature Importance

In [57]:
fitted_pipeline = final_model.regressor_

importance = pd.DataFrame({"feature": fitted_pipeline["preprocessing"].get_feature_names_out(),
                           "importance": fitted_pipeline["model"].feature_importances_}).sort_values("importance", ascending=False)
importance

,feature,importance
14,model,28.619037
16,year,25.144541
7,transmission_manual,10.730192
13,brand,9.118328
18,model_freq,5.932546
17,kilometers,5.459774
19,bm_first_year,4.527733
20,years_since_first_seen,4.004843
6,transmission_automatic,3.177776
15,gov,1.864726


# Conclusion
- After cleaning, 16,817 used cars were left from 17,159. We added 3 new features: model_freq, bm_first_year and years_since_first_seen.
- We compared XGBoost, LightGBM and CatBoost with 5-fold cross-validation. CatBoost was the best model.
- The model was trained on log(price), because prices range from 13K to 10M EGP.
- After tuning with RandomizedSearchCV and Optuna, the final model reached on the test set: R² (log) 94.58%, MAE 69,425 EGP, and a typical error of about 11%.
- R² in EGP is lower (81.84%) because of a few rare luxury cars that the model cannot price well.
- The most important features are model, year, transmission and brand.
- The model is saved with joblib and used in a Streamlit app that only shows car options that exist in the data.

# Save Model

In [58]:
joblib.dump(final_model, "final_model.pkl")

['final_model.pkl']

# Streamlit App

In [ ]:
import os

os.makedirs(".streamlit", exist_ok=True)
os.makedirs("app_pages", exist_ok=True)
# The car photos hero.jpg and banner.jpg must be inside static/
os.makedirs("static", exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#D97706"
backgroundColor = "#F5F7FB"
secondaryBackgroundColor = "#E9EEF6"
textColor = "#0B1F3A"
font = "sans serif"

[server]
# Serve the car photos in static/ at app/static/...
enableStaticServing = true

In [ ]:
%%writefile nile_common.py
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

BASE_DIR = Path(__file__).parent

# Median absolute % error of the final model on the test set (from the notebook)
TEST_MDAPE = 0.1142

# Car photos (Unsplash) stored in static/ and served by Streamlit (enableStaticServing in .streamlit/config.toml)
HERO_IMAGE = "app/static/hero.jpg"
BANNER_IMAGE = "app/static/banner.jpg"


@st.cache_data
def load_data():
    return pd.read_csv(BASE_DIR / "car_usage_clean.csv")


@st.cache_resource
def load_model():
    return joblib.load(BASE_DIR / "final_model.pkl")


# --- Display-only names: the model always receives the original values ---

def clean_name(name):
    """'Mazda 3 || Mazda 3' -> 'Mazda 3', 'Canghe || Changhe Q7' -> 'Changhe Q7'."""
    return name.replace("Canghe || ", "").split(" || ")[0]


@st.cache_data
def model_labels():
    """Clean label for every model; if two models of a brand would look the same, keep both parts."""
    models = load_data()[["brand", "model"]].drop_duplicates()
    models["label"] = models["model"].map(clean_name)
    clash = models.duplicated(["brand", "label"], keep=False)
    models.loc[clash, "label"] = models.loc[clash, "model"].str.replace(" || ", " / ", regex=False)
    return dict(zip(models["model"], models["label"]))


def capitalize(value):
    """'gas' -> 'Gas', 'natural gas' -> 'Natural gas', 'CVT' stays 'CVT'."""
    return value if value.isupper() else value[0].upper() + value[1:]


CSS = """
<style>
.block-container {padding-top: 2rem; max-width: 1150px;}

.nv-logo {font-size: 1.6rem; font-weight: 800; letter-spacing: -0.5px;}
.nv-logo .nile {color: #FFFFFF;}
.nv-logo .value {color: #F59E0B;}

.nv-hero {
    border-radius: 18px;
    padding: 3.2rem 3rem;
    color: #FFFFFF;
    background-color: #0B1F3A;
    background-image:
        linear-gradient(90deg, #0B1F3A 0%, #0B1F3A 32%, rgba(11,31,58,0.55) 52%, rgba(11,31,58,0.05) 100%),
        var(--hero-img);
    background-repeat: no-repeat;
    background-size: cover, 68% auto;
    background-position: center, right center;
    min-height: 330px;
}
.nv-hero h1 {color: #FFFFFF; font-size: 2.6rem; line-height: 1.15; margin: 1.2rem 0 0.8rem 0; max-width: 560px;}
.nv-hero p {color: #DCE4F2; font-size: 1.1rem; max-width: 520px; margin: 0;}

.nv-banner {
    border-radius: 18px;
    padding: 2.2rem 2.5rem;
    background-color: #0B1F3A;
    /* Same layout as the home hero: text on navy at the left, the whole car visible at the right */
    background-image:
        linear-gradient(90deg, #0B1F3A 0%, #0B1F3A 40%, rgba(11,31,58,0.5) 58%, rgba(11,31,58,0) 75%),
        var(--banner-img);
    background-repeat: no-repeat;
    background-size: cover, 60% auto;
    background-position: center, right 38%;
}
.nv-banner h1 {color: #FFFFFF; font-size: 2rem; margin: 0.6rem 0 0.3rem 0;}
.nv-banner p {color: #DCE4F2; margin: 0;}

.nv-card {
    background: #FFFFFF;
    border: 1px solid #E3E8F0;
    border-radius: 14px;
    padding: 1.4rem 1.5rem;
    height: 100%;
}
.nv-stat-number {font-size: 2rem; font-weight: 800; color: #0B1F3A;}
.nv-stat-label {color: #5B6B82; font-size: 0.95rem;}

.nv-step-number {
    display: inline-block; width: 34px; height: 34px; line-height: 34px;
    border-radius: 50%; background: #FEF3C7; color: #B45309;
    text-align: center; font-weight: 800; margin-bottom: 0.6rem;
}
.nv-step {min-height: 195px;}
.nv-step-title {font-weight: 700; font-size: 1.05rem; color: #0B1F3A;}
.nv-step-text {color: #5B6B82; font-size: 0.95rem;}

.nv-result {
    background: #0B1F3A;
    border-radius: 16px;
    padding: 1.8rem 2rem;
    color: #FFFFFF;
    text-align: center;
}
.nv-result-label {color: #B9C6DA; font-size: 0.95rem; text-transform: uppercase; letter-spacing: 1px;}
.nv-result-price {color: #F59E0B; font-size: 2.8rem; font-weight: 800; margin: 0.3rem 0;}
.nv-result-range {color: #DCE4F2; font-size: 1rem;}

.nv-section-title {font-size: 1.35rem; font-weight: 800; color: #0B1F3A; margin: 2rem 0 0.8rem 0;}
.nv-footer {color: #7A889E; font-size: 0.85rem; text-align: center; margin-top: 3rem;}
.nv-hero [data-testid="stHeaderActionElements"], .nv-banner [data-testid="stHeaderActionElements"] {display: none;}

/* Home call-to-action: bigger, bold and gently pulsing to catch the eye */
.st-key-nv_cta button {
    padding: 0.85rem 1.8rem;
    border-radius: 12px;
    animation: nv-pulse 1.8s ease-in-out infinite;
}
.st-key-nv_cta button p,
.st-key-nv_cta button span {font-size: 1.2rem; font-weight: 800;}
.st-key-nv_cta button:hover {animation-play-state: paused;}
@keyframes nv-pulse {
    0%   {transform: scale(1);    box-shadow: 0 0 0 0 rgba(217,119,6,0.55);}
    50%  {transform: scale(1.06); box-shadow: 0 0 0 14px rgba(217,119,6,0);}
    100% {transform: scale(1);    box-shadow: 0 0 0 0 rgba(217,119,6,0);}
}
/* Respect users who turned off animations in their system settings */
@media (prefers-reduced-motion: reduce) {
    .st-key-nv_cta button {animation: none;}
}

/* Phones and small tablets (kept last so it overrides the rules above) */
@media (max-width: 768px) {
    .nv-hero {
        padding: 2rem 1.5rem;
        min-height: 0;
        background-image: linear-gradient(rgba(11,31,58,0.78), rgba(11,31,58,0.78)), var(--hero-img);
        background-size: cover, cover;
        background-position: center, center;
    }
    .nv-hero h1 {font-size: 2rem;}
    .nv-banner {
        padding: 1.6rem 1.5rem;
        background-image: linear-gradient(rgba(11,31,58,0.78), rgba(11,31,58,0.78)), var(--banner-img);
        background-size: cover, cover;
        background-position: center, center 45%;
    }
    .nv-banner h1 {font-size: 1.6rem;}
    .nv-step {min-height: 215px;}
    .nv-result {padding: 1.4rem 1rem;}
    .nv-result-price {font-size: 2rem;}
    /* Market prices: 2 per row on phones instead of 4 stacked */
    .st-key-nv_market [data-testid="stHorizontalBlock"] {flex-wrap: wrap; gap: 1rem 0.75rem;}
    .st-key-nv_market [data-testid="stColumn"] {flex: 1 1 calc(50% - 0.75rem) !important; min-width: calc(50% - 0.75rem) !important;}
    .st-key-nv_market [data-testid="stMetricValue"] {font-size: 1.35rem;}
}
/* Phones: columns stack, so cards no longer need equal heights */
@media (max-width: 640px) {
    .nv-step {min-height: 0;}
}
</style>
"""


def apply_style():
    st.markdown(CSS, unsafe_allow_html=True)


def logo_html():
    return '<div class="nv-logo"><span class="nile">Nile</span><span class="value">Value</span></div>'


def footer():
    st.markdown(
        f'<div class="nv-footer">NileValue · Estimates are based on {len(load_data()):,} used car listings in Egypt '
        'and are for guidance only</div>',
        unsafe_allow_html=True,
    )

In [ ]:
%%writefile car_app.py
import streamlit as st

from nile_common import apply_style

st.set_page_config(page_title="NileValue | Used Car Prices in Egypt", page_icon="🚗", layout="wide")

apply_style()

home = st.Page("app_pages/home.py", title="Home", icon=":material/home:", default=True)
estimate = st.Page("app_pages/estimate.py", title="Estimate Price", icon=":material/sell:")

page = st.navigation([home, estimate], position="top")
page.run()

In [ ]:
%%writefile app_pages/home.py
from textwrap import dedent

import streamlit as st

from nile_common import HERO_IMAGE, TEST_MDAPE, footer, load_data, logo_html

df = load_data()

st.markdown(
    dedent(f"""
    <div class="nv-hero" style="--hero-img: url('{HERO_IMAGE}');">
        {logo_html()}
        <h1>Know your car's real price in Egypt</h1>
        <p>Data-driven price estimates for used cars, built on {len(df):,} real listings from the Egyptian market.</p>
    </div>
    """),
    unsafe_allow_html=True,
)

st.write("")
if st.container(key="nv_cta").button("Estimate my car's price", type="primary", icon=":material/arrow_forward:"):
    st.switch_page("app_pages/estimate.py")

st.markdown('<div class="nv-section-title">NileValue in numbers</div>', unsafe_allow_html=True)

stats = [
    (f"{len(df):,}", "Used car listings analysed"),
    (f"{df['brand'].nunique()} brands", f"{df['model'].nunique()} different models"),
    (f"~{TEST_MDAPE:.0%}", "Typical difference from the real price"),
]
for col, (number, label) in zip(st.columns(3), stats):
    col.markdown(
        f'<div class="nv-card"><div class="nv-stat-number">{number}</div>'
        f'<div class="nv-stat-label">{label}</div></div>',
        unsafe_allow_html=True,
    )

st.markdown('<div class="nv-section-title">How it works</div>', unsafe_allow_html=True)

steps = [
    ("Choose your car", "Pick the brand, model, year and fuel. Only cars that exist in the market data are shown."),
    ("Add the details", "Enter the transmission, color, kilometers and location of the car."),
    ("Get the price", "See the expected price, a typical price range and real prices of similar cars."),
]
for i, (col, (title, text)) in enumerate(zip(st.columns(3), steps), start=1):
    col.markdown(
        f'<div class="nv-card nv-step"><div class="nv-step-number">{i}</div>'
        f'<div class="nv-step-title">{title}</div>'
        f'<div class="nv-step-text">{text}</div></div>',
        unsafe_allow_html=True,
    )

footer()

In [ ]:
%%writefile app_pages/estimate.py
from textwrap import dedent

import pandas as pd
import streamlit as st

from nile_common import (BANNER_IMAGE, TEST_MDAPE, capitalize, clean_name, footer, load_data, load_model,
                         logo_html, model_labels)

df = load_data()
ml_model = load_model()
labels = model_labels()

st.markdown(
    dedent(f"""
    <div class="nv-banner" style="--banner-img: url('{BANNER_IMAGE}');">
        {logo_html()}
        <h1>Estimate your car's price</h1>
        <p>Fill in the car details, then press Estimate.</p>
    </div>
    """),
    unsafe_allow_html=True,
)

st.write("")

with st.container(border=True):
    col1, col2 = st.columns(2, gap="large")

    with col1:
        brand = st.selectbox("Brand", sorted(df["brand"].unique(), key=clean_name), format_func=clean_name)
        cars = df[df["brand"] == brand]

        model = st.selectbox("Model", sorted(cars["model"].unique(), key=labels.get), format_func=labels.get)
        cars = cars[cars["model"] == model]

        model_freq = len(cars)
        bm_first_year = cars["year"].min()

        year = st.selectbox("Year", sorted(cars["year"].unique(), reverse=True))
        cars = cars[cars["year"] == year]

        fuel = st.selectbox("Fuel", sorted(cars["fuel"].unique()), format_func=capitalize)
        cars = cars[cars["fuel"] == fuel]

    latest_year = df["year"].max()
    max_km = int(min((latest_year - year + 1) * 100_000, df["kilometers"].max()))
    # Start from the typical kilometers of this exact car in the data.
    # 0 km on old cars usually means "not filled in", so only recorded values are used;
    # if none exist, fall back to the typical kilometers of all cars from the same year.
    recorded_km = cars.loc[cars["kilometers"] > 0, "kilometers"]
    if recorded_km.empty:
        recorded_km = df.loc[(df["year"] == year) & (df["kilometers"] > 0), "kilometers"]
    default_km = int(min(round(recorded_km.median(), -3), max_km)) if not recorded_km.empty else 0

    with col2:
        transmission = st.selectbox("Transmission", sorted(cars["transmission"].unique()), format_func=capitalize)
        # Color and Location start from the most common value in the data
        colors = sorted(df["color"].unique())
        color = st.selectbox("Color", colors, index=colors.index(df["color"].mode()[0]))
        kilometers = st.number_input("Kilometers", min_value=0, max_value=max_km, value=default_km,
                                     step=5000, help=f"Maximum for a {year} car: {max_km:,} km")
        govs = sorted(df["gov"].unique())
        gov = st.selectbox("Location", govs, index=govs.index(df["gov"].mode()[0]))

    estimate = st.button("Estimate", type="primary", icon=":material/sell:")

if estimate:
    years_since_first_seen = year - bm_first_year
    new_data = pd.DataFrame(columns=df.columns.drop("price"),
                            data=[[brand, model, color, year, fuel, kilometers, transmission, gov,
                                   model_freq, bm_first_year, years_since_first_seen]])

    price = ml_model.predict(new_data)[0]
    low = round(price * (1 - TEST_MDAPE), -3)
    high = round(price * (1 + TEST_MDAPE), -3)

    st.markdown(
        dedent(f"""
        <div class="nv-result">
            <div class="nv-result-label">Expected price · {labels[model]} {year}</div>
            <div class="nv-result-price">{price:,.0f} EGP</div>
            <div class="nv-result-range">Typical range: {low:,.0f} – {high:,.0f} EGP</div>
        </div>
        """),
        unsafe_allow_html=True,
    )

    market = df[(df["model"] == model) & (df["year"] == year)]

    st.markdown(f'<div class="nv-section-title">Market prices · {labels[model]} {year}</div>', unsafe_allow_html=True)
    m1, m2, m3, m4 = st.container(key="nv_market").columns(4)
    if len(market) > 1:
        m1.metric("Average price", f"{market['price'].mean():,.0f} EGP")
        m2.metric("Lowest price", f"{market['price'].min():,.0f} EGP")
        m3.metric("Highest price", f"{market['price'].max():,.0f} EGP")
        m4.metric("Listings", f"{len(market):,}")
    else:
        # Only one listing: average, lowest and highest would all be the same number
        m1.metric("Listed price", f"{market['price'].iloc[0]:,.0f} EGP")
        m2.metric("Listings", "1")

    st.markdown('<div class="nv-section-title">Similar cars in the data</div>', unsafe_allow_html=True)
    similar = market.assign(km_diff=(market["kilometers"] - kilometers).abs()).sort_values("km_diff").head(5)
    similar = similar[["kilometers", "transmission", "fuel", "color", "gov", "price"]].assign(
        transmission=similar["transmission"].map(capitalize), fuel=similar["fuel"].map(capitalize))
    st.dataframe(
        similar,
        hide_index=True,
        width="stretch",
        column_config={
            "kilometers": st.column_config.NumberColumn("Kilometers", format="localized"),
            "transmission": "Transmission",
            "fuel": "Fuel",
            "color": "Color",
            "gov": "Location",
            "price": st.column_config.NumberColumn("Price (EGP)", format="localized"),
        },
    )

footer()

In [ ]:
! streamlit run car_app.py